# Notebook 1 - Del documento a los embeddings

Objetivo: comprender cómo un PDF se convierte en embeddings.

## Pipeline

```text
PDF
↓
LangChain Document
↓
Chunks
↓
Embeddings
↓
Búsqueda Semántica
```


## 1. Instalación

In [1]:
# !pip install langchain langchain-community langchain-text-splitters
# !pip install langchain-google-genai python-dotenv pypdf

## 2. Importaciones

In [2]:
from dotenv import load_dotenv
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings

C:\Users\Temporal\AppData\Local\Temp\ipykernel_27668\3607277215.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


| Librería | ¿Para qué sirve? |
|-----------|------------------|
| `dotenv` | Cargar la API Key desde el archivo `.env`. |
| `PyPDFLoader` | Leer un PDF y convertirlo en objetos `Document` de LangChain. |
| `RecursiveCharacterTextSplitter` | Dividir documentos largos en fragmentos más pequeños llamados **chunks**. |
| `GoogleGenerativeAIEmbeddings` | Convertir cada chunk de texto en un vector numérico (**embedding**). |

## 3. API Key

In [3]:
load_dotenv()
api_key=os.getenv("GEMINI_API_KEY")
print("API encontrada" if api_key else "No encontrada")

API encontrada


## 4. Configuración

In [4]:

embedding_model=GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=api_key
)

| Modelo | ¿Para qué se utiliza? |
|---------|-----------------------|
| `gemini-embedding-001` | Generar embeddings (vectores). |
| `gemini-2.5-flash` | Generar texto, responder preguntas, resumir, traducir, etc. |
| `gemini-2.5-pro` | Tareas complejas de razonamiento y generación de texto. |

| Modelo | Proveedor |
|---------|-----------|
| `gemini-embedding-001` | Google |
| `text-embedding-3-small` | OpenAI |
| `text-embedding-3-large` | OpenAI |
| `all-MiniLM-L6-v2` | Sentence Transformers |
| `bge-large-en` | BAAI |
| `e5-large-v2` | Microsoft |
| `varios` | Meta / Ollama* |

## 5. Cargar PDF

In [5]:
PDF_PATH="data/Renting.pdf"

loader=PyPDFLoader(PDF_PATH)
documents=loader.load()
print(len(documents))


6


In [6]:
type(documents[0])

langchain_core.documents.base.Document

## 6. Inspeccionar Document

In [7]:
documents[0]

Document(metadata={'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-27T00:43:07+02:00', 'author': 'Luis Umpire Alvarez', 'moddate': '2026-07-27T00:43:07+02:00', 'source': 'data/Renting.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Renting: \n*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación \npor parte del Banco. Oferta válida en Península y Baleares hasta el 30/10/2026. Oferta \nno válida para Ceuta y Melilla.  \n**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro \n20,99€ €/mes (IVA incluido). Se recibirá una bonificación de 20,99 € netos mensuales \n(tras aplicar la retención según normativa fiscal vigente, actualmente el 19 %) por la \ncontratación de un renting tecnológico a 36 meses para personas físicas que (1) \ndomicilien por primera vez su nómina o pensión superior a 1.200 € o cuota de \nautónomos o mutualidad y (2) la mantengan junto con l

¿Qué contiene `page_content` y qué contiene `metadata`?

In [8]:
documents[0].metadata

{'producer': 'Microsoft® Word LTSC',
 'creator': 'Microsoft® Word LTSC',
 'creationdate': '2026-07-27T00:43:07+02:00',
 'author': 'Luis Umpire Alvarez',
 'moddate': '2026-07-27T00:43:07+02:00',
 'source': 'data/Renting.pdf',
 'total_pages': 6,
 'page': 0,
 'page_label': '1'}

In [9]:
print(documents[0].metadata['source'])
print(documents[0].metadata['total_pages'])

data/Renting.pdf
6


In [10]:
documents[0].page_content

'Renting: \n*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación \npor parte del Banco. Oferta válida en Península y Baleares hasta el 30/10/2026. Oferta \nno válida para Ceuta y Melilla.  \n**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro \n20,99€ €/mes (IVA incluido). Se recibirá una bonificación de 20,99 € netos mensuales \n(tras aplicar la retención según normativa fiscal vigente, actualmente el 19 %) por la \ncontratación de un renting tecnológico a 36 meses para personas físicas que (1) \ndomicilien por primera vez su nómina o pensión superior a 1.200 € o cuota de \nautónomos o mutualidad y (2) la mantengan junto con la domiciliación de dos recibos \nmensuales, (3) un movimiento mensual de tarjeta de crédito o saldo en cuenta igual o \nsuperior a 1.000 € todos los días del mes y (4) tengan Bizum activo en Banco Iberico. Se \nrecibirá una bonificación de 27,99 € netos mensuales (tras aplicar la retención según \nn

## 7. Texto

In [11]:
print(documents[0].page_content[:1000])

Renting: 
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación 
por parte del Banco. Oferta válida en Península y Baleares hasta el 30/10/2026. Oferta 
no válida para Ceuta y Melilla.  
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 
20,99€ €/mes (IVA incluido). Se recibirá una bonificación de 20,99 € netos mensuales 
(tras aplicar la retención según normativa fiscal vigente, actualmente el 19 %) por la 
contratación de un renting tecnológico a 36 meses para personas físicas que (1) 
domicilien por primera vez su nómina o pensión superior a 1.200 € o cuota de 
autónomos o mutualidad y (2) la mantengan junto con la domiciliación de dos recibos 
mensuales, (3) un movimiento mensual de tarjeta de crédito o saldo en cuenta igual o 
superior a 1.000 € todos los días del mes y (4) tengan Bizum activo en Banco Iberico. Se 
recibirá una bonificación de 27,99 € netos mensuales (tras aplicar la retención según 
normativa fisca

## 9. Chunking

In [12]:
# Dividir el texto en fragmentos de 500 caracteres, dos fragmentos consecutivos comparten 100 carateres.
splitter=RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
# Se crea una lista
chunks=splitter.split_documents(documents)

print(len(chunks))

42


In [13]:
type(chunks)

list

## 10. Primer chunk

In [14]:
for i in range(3):
    print("=" * 80)
    print(f"Chunk {i}")
    print(f"Página original: {chunks[i].metadata['page']}")
    print(f"Número de caracteres: {len(chunks[i].page_content)}")
    print("-" * 80)
    print(chunks[i].page_content)
    print()

Chunk 0
Página original: 0
Número de caracteres: 476
--------------------------------------------------------------------------------
Renting: 
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación 
por parte del Banco. Oferta válida en Península y Baleares hasta el 30/10/2026. Oferta 
no válida para Ceuta y Melilla.  
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 
20,99€ €/mes (IVA incluido). Se recibirá una bonificación de 20,99 € netos mensuales 
(tras aplicar la retención según normativa fiscal vigente, actualmente el 19 %) por la

Chunk 1
Página original: 0
Número de caracteres: 416
--------------------------------------------------------------------------------
(tras aplicar la retención según normativa fiscal vigente, actualmente el 19 %) por la 
contratación de un renting tecnológico a 36 meses para personas físicas que (1) 
domicilien por primera vez su nómina o pensión superior a 1.200 € o cuota de 
autónom

## 11. Tamaños

In [15]:
print(f"Documento 0 (página completa) : {len(documents[0].page_content)} caracteres")
print(f"Chunk     0 (primer fragmento): {len(chunks[0].page_content)} caracteres")

Documento 0 (página completa) : 3073 caracteres
Chunk     0 (primer fragmento): 476 caracteres


## 12. Embeddings

In [16]:
embedding=embedding_model.embed_query(chunks[0].page_content)
print(type(embedding))
print(len(embedding))

<class 'list'>
3072


## Pregunta: ¿Por qué es una lista y no un vector?

## 13. Primeros valores

In [17]:
embedding[:20]

[0.0107669765,
 0.009831117,
 0.016420705,
 -0.06925308,
 0.017252732,
 0.026521415,
 0.013225581,
 -0.002256186,
 0.006123974,
 0.016981188,
 -0.03757811,
 0.012273067,
 0.00083024317,
 0.0019621914,
 0.12742142,
 0.047621656,
 -0.0034704516,
 0.004845018,
 0.0021173172,
 0.0022774122]

## Generamos todos los embeddings

In [18]:
embeddings = []

for chunk in chunks:
    embedding = embedding_model.embed_query(chunk.page_content)
    embeddings.append(embedding)

embeddings = [ Embedding 0, Embedding 1, Embedding 2, ... Embedding N ]

In [19]:
print(f"Número de embeddings generados: {len(embeddings)}")

Número de embeddings generados: 42


In [20]:
print(f"Dimensión del primer embedding: {len(embeddings[0])}")
print(f"Dimensión del último embedding: {len(embeddings[-1])}")

Dimensión del primer embedding: 3072
Dimensión del último embedding: 3072


## Comprobar la correspondencia entre chunks y embeddings

In [21]:
print(f"Número de chunks: {len(chunks)}")
print(f"Número de embeddings: {len(embeddings)}")

Número de chunks: 42
Número de embeddings: 42


- chunk 0 → embeeding 0
- chunk 1 → embeeding 1
- ...
- chunk n → embeeding n


```text
PDF
↓
Document
↓
Chunks
↓
Embeddings
```

Ya tenemos los textos representados en embedding, pero están en memoria:
- Se pierden al cerrar el programa.
- No podemos realizar búsquedas eficientes.
- No podemos almacenar miles o millones de embeddings.
- No podemos reutilizarlos en otro programa.

Necesitamos un sistema diseñado específicamente para almacenar y buscar vectores.

## ¿Dónde guardamos los embeddings?

La solución: una Base de Datos Vectorial

Una Base de Datos Vectorial (Vector Database) está optimizada para almacenar embeddings y realizar búsquedas por similitud.

En este curso utilizaremos **ChromaDB**

Su función dentro del pipeline será:


PDF

    ↓

Documents

    ↓

Chunks

    ↓

Embeddings

    ↓

ChromaDB


## ¿Qué información almacena ChromaDB?

Cada registro almacenado en ChromaDB contiene cuatro elementos principales:

| Campo | Descripción |
|--------|-------------|
| `id` | Identificador único del chunk. |
| `document` | Texto original del chunk. |
| `embedding` | Vector numérico asociado al chunk. |
| `metadata` | Información adicional del documento, como el archivo de origen, la página, el autor, etc. |

Visualmente, cada registro puede representarse de la siguiente forma:

```text
Registro

├── id
├── document
├── embedding
└── metadata
```

> **Idea clave:** ChromaDB no almacena únicamente los embeddings. También conserva el texto original (`document`) y su información asociada (`metadata`). Esto permitirá que, cuando recuperemos un embedding similar durante una búsqueda, podamos conocer el fragmento de texto del que proviene y mostrarlo como contexto al LLM.

Este concepto es fundamental y lo utilizaremos continuamente durante el resto del Sprint.

## Crear el cliente de ChromaDB

In [22]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_db")

### ¿Qué significa PersistentClient?

Le estamos diciendo a Chroma:

"Crea una base de datos en el disco duro y almacena allí toda la información."

La carpeta:

./chroma_db

se creará automáticamente la primera vez que ejecutemos el código.

Si vuelves a ejecutar el notebook mañana, los datos seguirán allí.

💡 
### Pensemos: Si Chroma ya tiene una base de datos, ¿qué creéis que almacenaremos primero: los embeddings o tendremos que crear antes algún tipo de contenedor?

## ¿Qué es una Collection?

Una **Collection** es el contenedor principal donde ChromaDB almacena los embeddings y la información asociada a ellos.

Si vienes del mundo de las bases de datos relacionales, puedes pensar en una **Collection** como el equivalente a una **tabla**.

| Base de Datos Relacional | ChromaDB |
|---------------------------|----------|
| Base de datos | PersistentClient |
| Tabla | Collection |
| Registro | Documento + Embedding + Metadata |
| Clave primaria | `id` |

Sin embargo, existe una diferencia importante.

En una base de datos relacional, una tabla almacena filas y columnas.

```text
Clientes

+----+---------+---------+
| id | nombre  | ciudad  |
+----+---------+---------+
| 1  | Ana     | Madrid  |
| 2  | Luis    | Sevilla |
+----+---------+---------+
```

En ChromaDB, una Collection almacena registros vectoriales.

```text
Collection

Registro 1
├── id
├── document
├── embedding
└── metadata

Registro 2
├── id
├── document
├── embedding
└── metadata
```

Cada Collection suele agrupar documentos relacionados con un mismo dominio o proyecto.

Por ejemplo:

- `manuales_empresa`
- `contratos`
- `noticias`
- `productos`
- `agenda_eventos`

En nuestro caso crearemos una Collection llamada:

```python
promo_iphone
```

Todos los chunks del documento se almacenarán dentro de esa Collection.

---

## ¿Por qué utilizar Collections?

Las Collections permiten organizar la información.

Por ejemplo, una empresa podría tener:

```text
ChromaDB

├── empleados
├── contratos
├── facturas
├── manuales
└── soporte_tecnico
```

Cada Collection contiene únicamente los documentos relacionados con ese tema.

De esta forma, cuando un usuario haga una consulta, podremos buscar únicamente en la Collection adecuada, mejorando el rendimiento y evitando recuperar información irrelevante.

---

## Idea clave

> Una Collection **no es un documento**, ni un embedding.
>
> Es un **contenedor** donde almacenamos muchos documentos vectorizados pertenecientes a un mismo dominio de conocimiento.

# Eliminando la Collection y Creando una nueva


In [23]:
try:
    client.delete_collection("renting")
except:
    pass

In [24]:
collection = client.create_collection(
    name="renting",
    metadata={"hnsw:space": "cosine"}
)

collection

Collection(name=renting)

**Nota:**
"Acabamos de crear la estructura. Todavía no hemos insertado ningún dato."

In [25]:
collection.count()

0

Similar a SQL:

CREATE TABLE clientes (...);

SELECT COUNT(*) FROM clientes;

# Preparando los datos para ChromaDB

## ¿Qué necesita ChromaDB para almacenar un chunk?

Hasta este momento ya disponemos de:

- Los **chunks** del documento.
- El **embedding** asociado a cada chunk.
- La **metadata** de cada chunk.

Sin embargo, ChromaDB no almacena únicamente los embeddings.

Cada registro que insertamos debe contener cuatro elementos:

| Elemento | Descripción |
|----------|-------------|
| `id` | Identificador único del chunk. |
| `document` | Texto original del chunk. |
| `embedding` | Vector numérico que representa el significado del chunk. |
| `metadata` | Información adicional del documento (archivo, página, autor, etc.). |

Podemos representarlo de la siguiente forma:

```text
Registro

├── id
├── document
├── embedding
└── metadata
```

Cada uno de nuestros chunks se convertirá en un registro dentro de la Collection.

```text
Chunk 0
        │
        ├── id
        ├── document
        ├── embedding
        └── metadata

Chunk 1
        │
        ├── id
        ├── document
        ├── embedding
        └── metadata

Chunk 2
        │
        ├── id
        ├── document
        ├── embedding
        └── metadata
```

Por tanto, antes de insertar los datos en ChromaDB debemos construir cuatro listas:

- Una lista con los identificadores (`ids`).
- Una lista con los documentos (`documents`).
- Una lista con los embeddings (`embeddings`).
- Una lista con la metadata (`metadatas`).

Estas cuatro listas tendrán exactamente el mismo número de elementos, ya que cada posición representa un mismo chunk.

Por ejemplo:

```text
Posición 0

ids[0]
documents[0]
embeddings[0]
metadatas[0]

↓

Registro 0
```

```text
Posición 1

ids[1]
documents[1]
embeddings[1]
metadatas[1]

↓

Registro 1
```

Esta correspondencia es fundamental, ya que ChromaDB utilizará el elemento de la posición `i` de cada lista para construir un único registro.

In [26]:
# Los identificadores únicos
ids = [f"chunk_{i}" for i in range(len(chunks))]

print(ids[:5])

['chunk_0', 'chunk_1', 'chunk_2', 'chunk_3', 'chunk_4']


In [27]:
# El texto de cada chunk
documents = [chunk.page_content for chunk in chunks]
print(documents[0])

Renting: 
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación 
por parte del Banco. Oferta válida en Península y Baleares hasta el 30/10/2026. Oferta 
no válida para Ceuta y Melilla.  
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 
20,99€ €/mes (IVA incluido). Se recibirá una bonificación de 20,99 € netos mensuales 
(tras aplicar la retención según normativa fiscal vigente, actualmente el 19 %) por la


In [28]:
metadatas = [chunk.metadata for chunk in chunks]

print(metadatas[0])

{'producer': 'Microsoft® Word LTSC', 'creator': 'Microsoft® Word LTSC', 'creationdate': '2026-07-27T00:43:07+02:00', 'author': 'Luis Umpire Alvarez', 'moddate': '2026-07-27T00:43:07+02:00', 'source': 'data/Renting.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}


Ya tenemos:
- Embeddings
- Ids
- chunks (documents)
- metadata

## ¿Es necesaria la metadata?

### Respuesta


- No
- Pero es útil para conocer el origen del documento y una posible ubicación, ejemplo:


Documentos

├── Contrato_A.pdf

├── Contrato_B.pdf

├── Manual_Técnico.pdf

├── Política_RRHH.pdf

└── Renting.pdf

- Documento : Contrato_A.pdf
- Página    : 18
- Categoría : Contratos

### Vamos a utilizar solo:
- ids
- documents
- embeddings

# Almacenando los datos en ChromaDB

## Insertando los registros en la Collection

Ya tenemos preparadas las tres listas necesarias:

- `ids`
- `documents`
- `embeddings`

Ahora utilizaremos el método `add()` para almacenar toda esa información dentro de la Collection.

```python
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings
)
```

Observa que no insertamos un registro cada vez.

En lugar de ello, ChromaDB recibe tres listas y construye automáticamente un registro utilizando los elementos que ocupan la misma posición en cada una de ellas.

Por ejemplo:

```text
Posición 0

ids[0]
documents[0]
embeddings[0]

↓

Registro 0
```

```text
Posición 1

ids[1]
documents[1]
embeddings[1]

↓

Registro 1
```

Es decir, ChromaDB recorre las tres listas en paralelo y crea un registro por cada índice.

Por ello, todas las listas deben tener exactamente el mismo número de elementos.

> **Nota:** ChromaDB también permite almacenar `metadata`, como el número de página o el documento de origen. En este primer ejemplo no la utilizaremos para centrarnos en el funcionamiento básico de una base de datos vectorial. Más adelante veremos cómo la metadata permite filtrar búsquedas y enriquecer los resultados.

In [29]:
collection.add(
    ids=ids,
    documents=documents,
    embeddings=embeddings
)

In [30]:
print(f"Número de registros almacenados: {collection.count()}")

Número de registros almacenados: 42


# Inspeccionando el contenido de la Collection: collection.get()

## Recuperando los registros almacenados

Ya hemos insertado todos los registros en la Collection.

Ahora vamos a comprobar qué información contiene utilizando el método `get()`.

```python
collection.get()
```

Este método recupera los registros almacenados en la Collection.

Si vienes del mundo de las bases de datos relacionales, puedes pensar en él como el equivalente a:

```sql
SELECT * FROM tabla;
```

La diferencia es que ChromaDB no devuelve una tabla, sino un diccionario de Python.

En ese diccionario encontraremos, entre otros, los siguientes elementos:

- `ids`
- `documents`
- `embeddings`

Cada posición de estas listas corresponde a un mismo registro almacenado en la Collection.

In [31]:
results = collection.get()

results

{'ids': ['chunk_0',
  'chunk_1',
  'chunk_2',
  'chunk_3',
  'chunk_4',
  'chunk_5',
  'chunk_6',
  'chunk_7',
  'chunk_8',
  'chunk_9',
  'chunk_10',
  'chunk_11',
  'chunk_12',
  'chunk_13',
  'chunk_14',
  'chunk_15',
  'chunk_16',
  'chunk_17',
  'chunk_18',
  'chunk_19',
  'chunk_20',
  'chunk_21',
  'chunk_22',
  'chunk_23',
  'chunk_24',
  'chunk_25',
  'chunk_26',
  'chunk_27',
  'chunk_28',
  'chunk_29',
  'chunk_30',
  'chunk_31',
  'chunk_32',
  'chunk_33',
  'chunk_34',
  'chunk_35',
  'chunk_36',
  'chunk_37',
  'chunk_38',
  'chunk_39',
  'chunk_40',
  'chunk_41'],
 'embeddings': None,
 'documents': ['Renting: \n*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación \npor parte del Banco. Oferta válida en Península y Baleares hasta el 30/10/2026. Oferta \nno válida para Ceuta y Melilla.  \n**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro \n20,99€ €/mes (IVA incluido). Se recibirá una bonificación de 20,99 €

Al ser un diccionario, utilizamos todo lo que sabemos hacer con diccionarios...

In [32]:
print(results.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])


In [33]:
print(results["ids"][:5])

['chunk_0', 'chunk_1', 'chunk_2', 'chunk_3', 'chunk_4']


In [34]:
print(results["documents"][0])

Renting: 
*Renting ofrecido por Banco Iberico. Operación de renting sujeta a previa aprobación 
por parte del Banco. Oferta válida en Península y Baleares hasta el 30/10/2026. Oferta 
no válida para Ceuta y Melilla.  
**Renting ofrecido por Banco Iberico. Renta mensual del iPhone 17e 256 GB Sin Seguro 
20,99€ €/mes (IVA incluido). Se recibirá una bonificación de 20,99 € netos mensuales 
(tras aplicar la retención según normativa fiscal vigente, actualmente el 19 %) por la


In [35]:
# ChromaDB no devuelve los embeddings por defecto. Los embeddings suelen ser vectores muy grandes
# print(len(results["embeddings"][0]))

In [36]:
results_2 = collection.get(
    include=["embeddings"]
)

In [37]:
print(len(results_2["embeddings"][0]))

3072


In [38]:
print((results_2["embeddings"][0][:10]))

[ 0.01076698  0.00983112  0.01642071 -0.06925308  0.01725273  0.02652142
  0.01322558 -0.00225619  0.00612397  0.01698119]


In [39]:
# util, obtener solo lo que necesitamos, por ejemplo:
results_2 = collection.get(
    include=["documents", "embeddings"]
)


### ¿Por qué `collection.get()` no devuelve los embeddings?

Los embeddings pueden contener cientos o miles de valores numéricos.

Si ChromaDB los devolviera automáticamente en cada consulta, el consumo de memoria sería mucho mayor.

Por este motivo, los embeddings solo se recuperan cuando se solicitan explícitamente mediante el parámetro `include`.

```python
results = collection.get(
    include=["embeddings"]
)
```

Este comportamiento mejora el rendimiento y evita transferir grandes cantidades de datos cuando no son necesarios.

## ¿Qué significan los demás campos?

Al ejecutar:

```python
results = collection.get()

print(results.keys())
```

observaremos que ChromaDB devuelve más información de la que hemos almacenado explícitamente.

```text
dict_keys([
    'ids',
    'documents',
    'embeddings',
    'metadatas',
    'uris',
    'data',
    'included'
])
```

Veamos brevemente para qué sirve cada uno de estos campos.

| Campo | Descripción |
|--------|-------------|
| `ids` | Identificadores únicos de los registros. |
| `documents` | Texto original almacenado. |
| `embeddings` | Vectores numéricos (solo si se solicitan mediante `include`). |
| `metadatas` | Información adicional asociada a cada documento. En este ejemplo no la utilizamos. |
| `uris` | Referencias a recursos externos (por ejemplo, archivos o ubicaciones URL). En este curso no las utilizaremos. |
| `data` | Campo opcional para almacenar información adicional, pe binaria. No lo utilizaremos en este curso. |
| `included` | Indica qué campos fueron solicitados mediante el parámetro `include` al realizar la consulta. |

En este curso nos centraremos únicamente en los tres elementos fundamentales de una base de datos vectorial:

- `ids`
- `documents`
- `embeddings`

Más adelante veremos cómo utilizar `metadatas` para enriquecer los documentos y realizar búsquedas filtradas.

# Realizando una búsqueda semántica

Hasta ahora hemos almacenado los embeddings dentro de ChromaDB.

La siguiente pregunta es:

> **¿Cómo encuentra ChromaDB los fragmentos de texto más relevantes para una consulta del usuario?**

La respuesta es sencilla:

1. El usuario escribe una pregunta en lenguaje natural.
2. Esa pregunta se convierte en un embedding utilizando el mismo modelo que utilizamos para los documentos.
3. ChromaDB compara ese embedding con los embeddings almacenados en la Collection.
4. Devuelve los documentos cuyos embeddings son más similares.

Visualmente:

```text
Pregunta del usuario
        │
        ▼
Embedding de la pregunta
        │
        ▼
Comparación con todos los embeddings
        │
        ▼
Embeddings más similares
        │
        ▼
Chunks recuperados
```

Este proceso recibe el nombre de **búsqueda semántica**, ya que no busca palabras exactamente iguales, sino fragmentos con un significado similar.

En este curso explicaremos la comparación utilizando la **similitud del coseno**, una de las métricas más utilizadas para medir la semejanza entre embeddings.

Cuanto más parecidos sean dos embeddings, mayor será su similitud y más probable será que representen textos con un significado similar.

# Ejemplo de uso: Usuario quiere tener un prompt para hacer una búsqueda

In [40]:
query = "¿Cuál es la duración mínima del contrato?"

query_embedding = embedding_model.embed_query(query)

In [41]:
print(type(query_embedding))
print(len(query_embedding))

<class 'list'>
3072


Chunks

      ↓

Embeddings


Pregunta

      ↓

Embedding

## Búsqueda: collection.query()
"Chroma ya no compara texto con texto. Compara embedding contra embedding."

In [42]:
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

In [43]:
print(results.keys())

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas', 'distances'])


In [45]:
print(collection.metadata)

{'hnsw:space': 'cosine'}


In [44]:
print(results["distances"])

[[0.34988248348236084, 0.3510127067565918, 0.35124391317367554]]


In [ ]:
distance = results["distances"][0][0]

similarity = 1 - distance  # Esto es válido sólo en ChromaDB

print(f"Distancia: {distance:.4f}")
print(f"Similitud: {similarity:.4f}")


Distancia: 0.3499
Similitud: 0.6501


In [50]:
for i in range(3):
    print("=" * 80)
    print(f"Resultado {i+1}")
    print(f"Similitud: {1- results['distances'][0][i]:.4f}")
    print()
    print(results["documents"][0][i])


Resultado 1
Similitud: 0.6501

cliente podrá adherirse a cualquier otra campaña vigente ya que esta campaña 
quedaría invalidada. 
 
El contrato de Renting debe estar al corriente de pago en el momento de la revisión 
mensual de las condiciones para poder recibir la bonificación en la fecha de pago de la 
misma. 2. Contratación (o ser ya titular) de una de las siguientes cuentas: Cuenta 
Iberico, Cuenta Online Iberico, Cuenta Negocios Iberico, Cuenta Colectivos, Cuenta One
Resultado 2
Similitud: 0.6490

(27,99 € netos1 ) mensuales por: Domiciliación de ingresos: nómina o pensión iguales o 
superiores a 3.000 €. A los efectos de los documentos de adhesión de la campaña se 
entenderá como pensión, cualquier tipo de prestación de carácter público y periódico. 
Se considerarán nóminas las percibidas por dichos conceptos a través del Sistema de 
Compensación Electrónico (SNCE) y pensión los pagos periódicos recibidos de la
Resultado 3
Similitud: 0.6488

revisión de las Condiciones de la cam

## Interpretando la similitud del coseno


```text
Similitud (Mejor) →  1
Similitud (Peor)  →  0
```



## NOTA:
Chroma utiliza una métrica de distancia. Si no especificas ninguna, usa la predeterminada de la versión instalada (en versiones recientes suele ser L2 o Euclidean Distance)